# Interactive 3D scatter explorer (scatter_cost)

Live alternative to `analyze.py`'s Plotly 3D plots. Uses `pandas` for the
median aggregation (fast enough on its own -- no out-of-core tooling needed)
and `k3d` (WebGL, actively maintained) for rendering, which handles far more
points smoothly than Plotly's inline-HTML `Scatter3d`.

(An earlier version used `ipyvolume` instead of `k3d` -- dropped after
hitting a GLSL shader compile error ["undeclared identifier"] in its
`pythreejs`-based renderer, a real bug in that unmaintained dependency
(no release since ~2021), not fixable from notebook code.)

Reads `results_agg.csv` (produced once by `doit agg`, see dodo.py) instead of
re-parsing the raw per-iteration `results.csv` and re-computing the median on
every notebook restart -- run `doit agg` first (or after regenerating
`results.csv`) if it's missing or stale.

Filtering works by toggling each point's size to 0 rather than removing it
from the arrays -- avoids resizing the GPU buffer on every slider drag, which
is what keeps this smooth even with millions of points.

Run with `jupyter lab`, or open directly in VS Code's notebook editor.


In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, str(pathlib.Path.cwd()))
from analyze import (
    PROBLEM,
    LATENCY,
    _log_ticks,
    principal_dims,
)  # reuse Dim descriptors + log-tick helper
from matplotlib.ticker import MaxNLocator

AGG_CSV = "plots/sg/results_agg.csv"  # produced by `doit agg`; point at your sweep

agg = pd.read_csv(AGG_CSV)
print(f"{len(agg):,} points")

248,158 points


In [2]:
import k3d
import k3d.colormaps.matplotlib_color_maps as mcm

x_dim, y_dim = principal_dims(agg)
z_dim = LATENCY
color_dim = PROBLEM.group

# k3d renders positions as literal world-space coordinates with a single
# shared scale across all 3 axes -- unlike Plotly's 3D scene, it can't give
# each axis its own independent domain. block_size/latency's *log*-space
# range (a few units) is naturally much narrower than blocks_per_dpu's raw
# range (1-24), so without rescaling, latency/block_size look squashed flat.
# Fix: independently normalize each axis's (possibly log-transformed) values
# to the same [0, AXIS_SPAN] span, then place real-value tick labels (not
# k3d's own, which would just show the meaningless normalized coordinate) at
# the corresponding normalized position.
AXIS_SPAN = 10.0


def to_axis_space(dim, real_values):
    v = np.asarray(real_values, dtype=float)
    if dim.log == 2:
        return np.log2(v)
    if dim.log == 10:
        return np.log10(v)
    return v


def axis_label(dim):
    def escape(s):
        return s.replace("_", "\\_")

    return (
        f"\\operatorname{{log}}_{{{dim.log}}}(\\mathrm{{{escape(str(dim.col))}}})"
        if dim.log
        else f"\\mathrm{{{escape(dim.col)}}}"
    )


def nice_ticks(dim, real_values):
    # Real-valued tick marks for `dim`: nice log-spaced powers for a
    # log-scaled dim (matching analyze.py's own colorbar/axis ticks), nice
    # round linear numbers otherwise.
    if dim.log is not None:
        return _log_ticks(real_values, dim.log)
    lo, hi = float(np.min(real_values)), float(np.max(real_values))
    return [t for t in MaxNLocator(nbins=6).tick_values(lo, hi) if lo <= t <= hi]


class Axis:
    # One plotted dimension: its own (lo, hi) in log-space (if any) and the
    # normalize() mapping real values -> this axis's shared [0, AXIS_SPAN]
    # world-space coordinate.
    def __init__(self, dim, real_values, span=AXIS_SPAN):
        self.dim = dim
        self.span = span
        space = to_axis_space(dim, real_values)
        self.lo, self.hi = float(space.min()), float(space.max())

    def normalize(self, real_values):
        space = to_axis_space(self.dim, real_values)
        return (space - self.lo) / (self.hi - self.lo) * self.span


x_axis = Axis(x_dim, agg[x_dim.col])
y_axis = Axis(y_dim, agg[y_dim.col])
z_axis = Axis(z_dim, agg[z_dim.col], span=AXIS_SPAN / 2)

x, y, z = (
    x_axis.normalize(agg[x_dim.col]),
    y_axis.normalize(agg[y_dim.col]),
    z_axis.normalize(agg[z_dim.col]),
)
positions = np.column_stack([x, y, z]).astype(np.float32)
attr = to_axis_space(color_dim, agg[color_dim.col]).astype(np.float32)

BASE_SIZE = 0.05

points = k3d.points(
    positions=positions,
    attribute=attr,
    color_map=mcm.Viridis,
    color_range=[float(attr.min()), float(attr.max())],
    point_sizes=np.full(len(agg), BASE_SIZE, dtype=np.float32),
    # "flat" is a cheap camera-facing billboard shader -- handles far more
    # points smoothly than the default "3dSpecular" (lit 3D mesh per point).
    shader="flat",
)

plot = k3d.plot()
plot.axes = [axis_label(x_dim), axis_label(y_dim), axis_label(z_dim)]
plot += points

# Every axis now spans the same [0, AXIS_SPAN] world-space range, so 0 is
# every axis's own low edge -- place each dim's own tick labels along the
# edge where the other two axes sit at their low edge.
for label in [
    k3d.text(
        f"{real_val:g}",
        position=[x_axis.normalize([real_val])[0], 0, 0],
        color=0,
        size=0.8,
    )
    for real_val in nice_ticks(x_dim, agg[x_dim.col])
]:
    plot += label
for label in [
    k3d.text(
        f"{real_val:g}",
        position=[0, y_axis.normalize([real_val])[0], 0],
        color=0,
        size=0.8,
    )
    for real_val in nice_ticks(y_dim, agg[y_dim.col])
]:
    plot += label
for label in [
    k3d.text(
        f"{real_val:g}",
        position=[0, 0, z_axis.normalize([real_val])[0]],
        color=0,
        size=0.8,
    )
    for real_val in nice_ticks(z_dim, agg[z_dim.col])
]:
    plot += label


def make_slider(dim):
    values = sorted(int(v) for v in agg[dim.col].unique())
    return widgets.SelectionRangeSlider(
        options=values,
        index=(0, len(values) - 1),
        description=dim.col,
        continuous_update=True,
        layout=widgets.Layout(width="500px"),
    )


sliders = {d.col: make_slider(d) for d in PROBLEM.all_dims}


def update(*_):
    mask = np.ones(len(agg), dtype=bool)
    for d in PROBLEM.all_dims:
        lo, hi = sliders[d.col].value
        col = agg[d.col].to_numpy()
        mask &= (col >= lo) & (col <= hi)
    points.point_sizes = np.where(mask, BASE_SIZE, 0.0).astype(np.float32)


for s in sliders.values():
    s.observe(update, names="value")

display(widgets.VBox(list(sliders.values())))
plot.display()

Output()

# MLP overlay: generalization on the full dense grid

Loads the MLP weights dumped by `analyze.py --mlp` (see `export_mlp_npz`) next
to `AGG_CSV`, and predicts latency over the *full* dense sweep grid, not just
the rows actually present in `agg`:

`scatter_bench.cpp`'s `sampleFairLog2` (see `genDpuCounts`/`genBlockSizes`/
`genBlocksPerDpu`) sweeps `num_dpus`, `blocks_per_dpu` and `block_size` each on
its own independent grid -- so each axis's *observed* unique values in `agg`
already **are** that per-axis grid, and the cartesian product of the three is
the full dense sweep. No need to reimplement `sampleFairLog2` in Python: rows
missing from `agg` (e.g. `blocks_per_dpu * block_size` too large for a DPU's
MRAM) are exactly the combos this predicts for, to see whether the MLP
generalizes sanely into the gaps.

Rendered as:
- a point cloud at every grid combo's predicted latency, colored dark gray if
  that combo was actually observed, bright orange if it wasn't;
- a wireframe mesh connecting each point to its immediate `x_dim`/`y_dim`
  neighbor, built independently per `num_dpus` value (never linking two
  different `num_dpus` slices together).

Both react to the same range sliders as the measured scatter above.

Run `analyze.py <csv> --mlp` first if `mlp_weights.npz` doesn't exist yet
next to `results_agg.csv`.

In [3]:
MLP_PATH = pathlib.Path(AGG_CSV).parent / "mlp_weights.npz"

if not MLP_PATH.exists():
    print(f"{MLP_PATH} not found -- run analyze.py on this CSV with --mlp first")
else:
    npz = np.load(MLP_PATH, allow_pickle=True)
    mlp_feature_names = [str(s) for s in npz["feature_names"]]
    mlp_feature_transforms = [(str(s) or None) for s in npz["feature_transforms"]]
    mlp_n_layers = int(npz["n_layers"])
    mlp_weights = [npz[f"weight_{i}"] for i in range(mlp_n_layers)]
    mlp_biases = [npz[f"bias_{i}"] for i in range(mlp_n_layers)]
    mlp_mean, mlp_std = npz["mean"], npz["std"]

    def mlp_predict_ms(df):
        # Same feature transform + standardization + forward pass as
        # export_mlp_cpp's generated C++ / fit_mlp's training, replayed here
        # in plain numpy against the dumped weights.
        cols = [
            np.log2(df[name].to_numpy(dtype=float))
            if transform == "log2"
            else df[name].to_numpy(dtype=float)
            for name, transform in zip(mlp_feature_names, mlp_feature_transforms)
        ]
        h = (np.column_stack(cols) - mlp_mean) / mlp_std
        for i, (w, b) in enumerate(zip(mlp_weights, mlp_biases)):
            h = h @ w + b
            if i < mlp_n_layers - 1:
                h = np.maximum(h, 0.0)  # ReLU
        return np.exp(h[:, 0])  # undo fit_mlp's log-space target

    # See the markdown above: each axis's own observed unique values already
    # are scatter_bench.cpp's full per-axis sampleFairLog2 grid, so the
    # cartesian product of the three is the full dense sweep -- no need to
    # reimplement sampleFairLog2 here.
    group_vals = sorted(agg[color_dim.col].unique())
    x_vals = sorted(agg[x_dim.col].unique())
    y_vals = sorted(agg[y_dim.col].unique())
    n_group, n_x, n_y = len(group_vals), len(x_vals), len(y_vals)

    # Row order matches idx.reshape(n_group, n_x, n_y) further down (group
    # slowest-varying, y fastest) -- required for the vectorized wireframe
    # edges below to line up with grid_df's rows.
    grid_df = pd.MultiIndex.from_product(
        [group_vals, x_vals, y_vals],
        names=[color_dim.col, x_dim.col, y_dim.col],
    ).to_frame(index=False)

    observed_index = agg.set_index([color_dim.col, x_dim.col, y_dim.col]).index
    grid_df["observed"] = grid_df.set_index(
        [color_dim.col, x_dim.col, y_dim.col]
    ).index.isin(observed_index)
    grid_df["pred_ms"] = mlp_predict_ms(grid_df)

    n_unobserved = int((~grid_df["observed"]).sum())
    print(
        f"MLP grid: {len(grid_df):,} combos total "
        f"({len(grid_df) - n_unobserved:,} observed, {n_unobserved:,} unobserved)"
    )

    OBSERVED_COLOR = 0x202020
    UNOBSERVED_COLOR = 0xFF7F0E  # bright orange -- extrapolated points stand out

    grid_positions = np.column_stack(
        [
            x_axis.normalize(grid_df[x_dim.col]),
            y_axis.normalize(grid_df[y_dim.col]),
            z_axis.normalize(grid_df["pred_ms"]),
        ]
    ).astype(np.float32)
    grid_colors = np.where(
        grid_df["observed"].to_numpy(), OBSERVED_COLOR, UNOBSERVED_COLOR
    ).astype(np.uint32)
    GRID_POINT_SIZE = BASE_SIZE * 0.6

    grid_points = k3d.points(
        positions=grid_positions,
        colors=grid_colors,
        point_sizes=np.full(len(grid_df), GRID_POINT_SIZE, dtype=np.float32),
        shader="flat",
    )
    plot += grid_points

    # Wireframe edges connecting each grid point to its immediate x_dim/y_dim
    # neighbor, one gapless (x_dim, y_dim) grid per group_dim (num_dpus)
    # value -- built via reshape/slicing (not a per-group Python loop) so it
    # stays fast even at ~600k vertices / ~1.2M edges; group_dim slices are
    # never linked to each other since edges only ever pair indices sharing
    # the same leading (group) index in the reshape. k3d stores `indices` as
    # float32 (not an int dtype) despite the name, hence the cast.
    idx = np.arange(len(grid_df)).reshape(n_group, n_x, n_y)
    x_edges = np.stack([idx[:, :-1, :].ravel(), idx[:, 1:, :].ravel()], axis=1)
    y_edges = np.stack([idx[:, :, :-1].ravel(), idx[:, :, 1:].ravel()], axis=1)
    grid_segments = np.concatenate([x_edges, y_edges], axis=0)

    wireframe = k3d.lines(
        vertices=grid_positions,
        indices=grid_segments.astype(np.float32),
        indices_type="segment",
        color=OBSERVED_COLOR,
        shader="simple",
        width=0.01,
    )
    plot += wireframe

    print(f"MLP grid wireframe: {len(grid_segments):,} edges")

    # React to the same range sliders as the measured scatter above (see
    # `sliders`/`update` in the previous cell): toggle grid_points' sizes the
    # same way, and re-filter grid_segments down to edges whose *both*
    # endpoints still pass every slider's range -- cheap boolean indexing
    # even at ~1.2M edges, so this stays interactive.
    def mlp_grid_mask():
        mask = np.ones(len(grid_df), dtype=bool)
        for d in PROBLEM.all_dims:
            lo, hi = sliders[d.col].value
            col = grid_df[d.col].to_numpy()
            mask &= (col >= lo) & (col <= hi)
        return mask

    def update_mlp_grid(*_):
        mask = mlp_grid_mask()
        grid_points.point_sizes = np.where(mask, GRID_POINT_SIZE, 0.0).astype(
            np.float32
        )
        edge_mask = mask[grid_segments[:, 0]] & mask[grid_segments[:, 1]]
        wireframe.indices = grid_segments[edge_mask].astype(np.float32)

    for s in sliders.values():
        s.observe(update_mlp_grid, names="value")

MLP grid: 606,171 combos total (248,158 observed, 358,013 unobserved)
MLP grid wireframe: 1,192,784 edges
